# ChaiAnalytics — Model-Native Regression Benchmark

Interactive notebook for exploring and visualizing model evaluation results.

**Models evaluated in their native paradigms:**
- **SARIMA**: Expanding-window time-series CV (forecast horizons 1, 3, 6 months)
- **XGBoost**: TimeSeriesSplit CV + holdout test set (2023-2024)
- **GNN**: Cluster quality metrics + anomaly score analysis

**Metrics**: RMSE, MSE, R², MAE, MAPE

In [3]:
# Setup
import sys
sys.path.insert(0, '../scripts')

import os
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

BASE_DIR = Path('../').resolve()
MODELS_DIR = BASE_DIR / 'models'
RESULTS_DIR = BASE_DIR / 'results' / 'benchmarks'

print(f'Base dir: {BASE_DIR}')
print(f'Models dir: {MODELS_DIR}')

Base dir: /home/mogalka/projects/chai-analytics
Models dir: /home/mogalka/projects/chai-analytics/models


## 1. Load Existing Benchmark Results (if available)

In [4]:
# Find latest benchmark results
benchmark_dirs = sorted(RESULTS_DIR.glob('*')) if RESULTS_DIR.exists() else []

if benchmark_dirs:
    latest = benchmark_dirs[-1]
    print(f'Latest benchmark: {latest}')
    
    # Load summary
    summary_path = latest / 'benchmark_summary.json'
    if summary_path.exists():
        with open(summary_path) as f:
            summary = json.load(f)
        print(json.dumps(summary, indent=2, default=str))
    
    # Load detailed CSVs
    sarima_df = pd.read_csv(latest / 'sarima_detailed.csv') if (latest / 'sarima_detailed.csv').exists() else None
    xgb_cv_df = pd.read_csv(latest / 'xgb_cv_detailed.csv') if (latest / 'xgb_cv_detailed.csv').exists() else None
    xgb_farm_df = pd.read_csv(latest / 'xgb_per_farm.csv') if (latest / 'xgb_per_farm.csv').exists() else None
    xgb_pred_df = pd.read_csv(latest / 'xgb_predictions.csv') if (latest / 'xgb_predictions.csv').exists() else None
    gnn_cluster_df = pd.read_csv(latest / 'gnn_per_cluster.csv') if (latest / 'gnn_per_cluster.csv').exists() else None
    
    print(f'\nLoaded dataframes:')
    for name, df in [('SARIMA', sarima_df), ('XGB_CV', xgb_cv_df), ('XGB_Farm', xgb_farm_df), ('XGB_Pred', xgb_pred_df), ('GNN_Cluster', gnn_cluster_df)]:
        if df is not None:
            print(f'  {name}: {df.shape}')
else:
    print('No previous benchmark results found. Run the script first:')
    print('  python scripts/benchmark_regression.py')

No previous benchmark results found. Run the script first:
  python scripts/benchmark_regression.py


## 2. Run Benchmark (if not already run)

In [11]:
# Run the benchmark script
import subprocess
result = subprocess.run([sys.executable, '../scripts/benchmark_regression.py'], 
                        capture_output=True, text=True, cwd=BASE_DIR)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)


STDERR: /home/mogalka/projects/chai-analytics/venv/bin/python: can't open file '/home/mogalka/projects/chai-analytics/../scripts/benchmark_regression.py': [Errno 2] No such file or directory



## 3. SARIMA — Expanding Window Time-Series CV Analysis

In [6]:
# SARIMA Results Visualization
if sarima_df is not None and len(sarima_df) > 0:
    print(f'SARIMA Results: {len(sarima_df)} evaluation points')
    print(f'Farms evaluated: {sarima_df["member_no"].nunique()}')
    print(f'Horizons: {sorted(sarima_df["horizon"].unique())}')
    print(f'Test seasons: {sorted(sarima_df["test_season"].unique())}')
    
    # Aggregate by horizon
    horizon_agg = sarima_df.groupby('horizon')[['rmse', 'mse', 'r2', 'mae', 'mape']].agg(['mean', 'std', 'count'])
    print('\nAggregate by horizon:')
    display(horizon_agg.round(2))
    
    # Plot RMSE by horizon
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # RMSE
    ax = axes[0, 0]
    for h in sorted(sarima_df['horizon'].unique()):
        data = sarima_df[sarima_df['horizon'] == h]['rmse']
        ax.boxplot(data, positions=[h], widths=0.5, label=f'h={h}')
    ax.set_xlabel('Forecast Horizon (months)')
    ax.set_ylabel('RMSE (kg)')
    ax.set_title('SARIMA RMSE by Forecast Horizon')
    ax.legend()
    
    # R²
    ax = axes[0, 1]
    for h in sorted(sarima_df['horizon'].unique()):
        data = sarima_df[sarima_df['horizon'] == h]['r2'].dropna()
        if len(data) > 0:
            ax.boxplot(data, positions=[h], widths=0.5, label=f'h={h}')
    ax.set_xlabel('Forecast Horizon (months)')
    ax.set_ylabel('R²')
    ax.set_title('SARIMA R² by Forecast Horizon')
    ax.axhline(y=0, color='r', linestyle='--', alpha=0.5)
    ax.legend()
    
    # MAE
    ax = axes[1, 0]
    for h in sorted(sarima_df['horizon'].unique()):
        data = sarima_df[sarima_df['horizon'] == h]['mae']
        ax.boxplot(data, positions=[h], widths=0.5, label=f'h={h}')
    ax.set_xlabel('Forecast Horizon (months)')
    ax.set_ylabel('MAE (kg)')
    ax.set_title('SARIMA MAE by Forecast Horizon')
    ax.legend()
    
    # MAPE
    ax = axes[1, 1]
    for h in sorted(sarima_df['horizon'].unique()):
        data = sarima_df[sarima_df['horizon'] == h]['mape'].dropna()
        if len(data) > 0:
            ax.boxplot(data, positions=[h], widths=0.5, label=f'h={h}')
    ax.set_xlabel('Forecast Horizon (months)')
    ax.set_ylabel('MAPE (%)')
    ax.set_title('SARIMA MAPE by Forecast Horizon')
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Per-farm performance
    farm_agg = sarima_df.groupby('member_no')[['rmse', 'r2', 'mae']].mean().reset_index()
    farm_agg = farm_agg.sort_values('rmse')
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(farm_agg['rmse'], bins=30, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Mean RMSE (kg)')
    axes[0].set_ylabel('Number of Farms')
    axes[0].set_title('Distribution of Per-Farm Mean RMSE')
    
    axes[1].hist(farm_agg['r2'].dropna(), bins=30, edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('Mean R²')
    axes[1].set_ylabel('Number of Farms')
    axes[1].set_title('Distribution of Per-Farm Mean R²')
    
    plt.tight_layout()
    plt.show()
    
    # Top/Bottom farms
    print('\nBest 10 farms (lowest RMSE):')
    display(farm_agg.head(10))
    print('\nWorst 10 farms (highest RMSE):')
    display(farm_agg.tail(10))
else:
    print('No SARIMA results available')

NameError: name 'sarima_df' is not defined

## 4. XGBoost — TimeSeriesSplit CV + Holdout Analysis

In [8]:
# XGBoost Results Visualization
if xgb_cv_df is not None:
    print('XGBoost CV Results:')
    display(xgb_cv_df.round(4))
    
    # CV metrics across folds
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    metrics = ['rmse', 'r2', 'mae', 'mape']
    titles = ['RMSE (kg)', 'R²', 'MAE (kg)', 'MAPE (%)']
    
    for ax, metric, title in zip(axes.flat, metrics, titles):
        ax.bar(xgb_cv_df['fold'], xgb_cv_df[metric], alpha=0.7, edgecolor='black')
        ax.axhline(y=xgb_cv_df[metric].mean(), color='red', linestyle='--', 
                   label=f'Mean: {xgb_cv_df[metric].mean():.3f}')
        ax.set_xlabel('Fold')
        ax.set_ylabel(title)
        ax.set_title(f'XGBoost CV: {title} by Fold')
        ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    # CV summary
    cv_summary = xgb_cv_df[['rmse', 'mse', 'r2', 'mae', 'mape']].agg(['mean', 'std'])
    print('\nCV Summary:')
    display(cv_summary.round(4))
else:
    print('No XGBoost CV results available')

NameError: name 'xgb_cv_df' is not defined

In [9]:
# XGBoost Holdout Test Set Analysis
if xgb_pred_df is not None:
    print(f'Holdout predictions: {len(xgb_pred_df)} rows')
    print(f'Test seasons: {sorted(xgb_pred_df["season_year"].unique())}')
    
    # Actual vs Predicted scatter
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Scatter plot
    ax = axes[0, 0]
    ax.scatter(xgb_pred_df['actual'], xgb_pred_df['predicted'], alpha=0.5, s=10)
    max_val = max(xgb_pred_df['actual'].max(), xgb_pred_df['predicted'].max())
    ax.plot([0, max_val], [0, max_val], 'r--', label='Perfect prediction')
    ax.set_xlabel('Actual (kg)')
    ax.set_ylabel('Predicted (kg)')
    ax.set_title('XGBoost: Actual vs Predicted (Holdout)')
    ax.legend()
    ax.set_aspect('equal')
    
    # Residuals
    ax = axes[0, 1]
    ax.scatter(xgb_pred_df['predicted'], xgb_pred_df['error'], alpha=0.5, s=10)
    ax.axhline(y=0, color='r', linestyle='--')
    ax.set_xlabel('Predicted (kg)')
    ax.set_ylabel('Residual (kg)')
    ax.set_title('XGBoost: Residuals vs Predicted')
    
    # Error distribution
    ax = axes[1, 0]
    ax.hist(xgb_pred_df['error'], bins=50, edgecolor='black', alpha=0.7)
    ax.set_xlabel('Error (kg)')
    ax.set_ylabel('Frequency')
    ax.set_title('XGBoost: Error Distribution')
    
    # MAPE by actual yield bins
    ax = axes[1, 1]
    valid = xgb_pred_df.dropna(subset=['ape'])
    valid['yield_bin'] = pd.qcut(valid['actual'], q=10, duplicates='drop')
    mape_by_bin = valid.groupby('yield_bin')['ape'].mean()
    mape_by_bin.plot(kind='bar', ax=ax, edgecolor='black')
    ax.set_xlabel('Actual Yield Bin')
    ax.set_ylabel('Mean MAPE (%)')
    ax.set_title('XGBoost: MAPE by Yield Level')
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Per-farm analysis
    if xgb_farm_df is not None:
        print('\nPer-farm error distribution:')
        display(xgb_farm_df[['mae', 'rmse', 'mape']].describe().round(2))
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].hist(xgb_farm_df['mae'], bins=30, edgecolor='black', alpha=0.7)
        axes[0].set_xlabel('Per-Farm MAE (kg)')
        axes[0].set_ylabel('Count')
        axes[0].set_title('Distribution of Per-Farm MAE')
        
        axes[1].hist(xgb_farm_df['mape'].dropna(), bins=30, edgecolor='black', alpha=0.7)
        axes[1].set_xlabel('Per-Farm MAPE (%)')
        axes[1].set_ylabel('Count')
        axes[1].set_title('Distribution of Per-Farm MAPE')
        plt.tight_layout()
        plt.show()
        
        # Worst farms
        print('\nWorst 10 farms by MAE:')
        display(xgb_farm_df.nlargest(10, 'mae')[['ktda_member_no', 'n_obs', 'mae', 'rmse', 'mape']])
else:
    print('No XGBoost holdout results available')

NameError: name 'xgb_pred_df' is not defined

## 5. GNN — Cluster Quality + Anomaly Score Analysis

In [10]:
# GNN Results Visualization
if gnn_cluster_df is not None:
    print('GNN Cluster Summary:')
    display(gnn_cluster_df)
    
    # Load full GNN outputs for detailed analysis
    from pymongo import MongoClient
    MONGODB_URI = os.getenv('MONGODB_URI', 'mongodb://localhost:27017/chaimterics')
    client = MongoClient(MONGODB_URI, serverSelectionTimeoutMS=5000)
    db = client.get_default_database()
    
    gnn_outputs = pd.DataFrame(list(db.gnn_outputs.find({}, {'_id': 0})))
    print(f'\nTotal farms with GNN outputs: {len(gnn_outputs)}')
    
    # Anomaly score distribution
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Histogram
    ax = axes[0, 0]
    ax.hist(gnn_outputs['anomaly_score'], bins=40, edgecolor='black', alpha=0.7)
    ax.axvline(gnn_outputs['anomaly_score'].mean(), color='red', linestyle='--', 
               label=f'Mean: {gnn_outputs["anomaly_score"].mean():.3f}')
    ax.axvline(0.7, color='orange', linestyle='--', label='Threshold 0.7')
    ax.axvline(0.5, color='green', linestyle='--', label='Threshold 0.5')
    ax.set_xlabel('Anomaly Score')
    ax.set_ylabel('Number of Farms')
    ax.set_title('GNN: Anomaly Score Distribution')
    ax.legend()
    
    # Anomaly by cluster
    ax = axes[0, 1]
    cluster_order = sorted(gnn_outputs['cluster_id'].unique())
    bp = ax.boxplot([gnn_outputs[gnn_outputs['cluster_id']==c]['anomaly_score'] 
                    for c in cluster_order], 
                    labels=[f'Cluster {c}' for c in cluster_order])
    ax.set_xlabel('Cluster')
    ax.set_ylabel('Anomaly Score')
    ax.set_title('GNN: Anomaly Scores by Cluster')
    
    # Cluster sizes
    ax = axes[1, 0]
    cluster_counts = gnn_outputs['cluster_id'].value_counts().sort_index()
    ax.bar(cluster_counts.index, cluster_counts.values, edgecolor='black', alpha=0.7)
    ax.set_xlabel('Cluster ID')
    ax.set_ylabel('Number of Farms')
    ax.set_title('GNN: Cluster Sizes')
    
    # Top anomalous farms
    ax = axes[1, 1]
    top_anomalies = gnn_outputs.nlargest(15, 'anomaly_score')
    ax.barh(range(len(top_anomalies)), top_anomalies['anomaly_score'], edgecolor='black', alpha=0.7)
    ax.set_yticks(range(len(top_anomalies)))
    ax.set_yticklabels([f"{row['ktda_member_no']} (C{row['cluster_id']})" 
                       for _, row in top_anomalies.iterrows()])
    ax.set_xlabel('Anomaly Score')
    ax.set_title('GNN: Top 15 Anomalous Farms')
    ax.invert_yaxis()
    
    plt.tight_layout()
    plt.show()
    
    # Summary stats
    print(f'\nAnomaly Score Stats:')
    print(f'  Mean: {gnn_outputs["anomaly_score"].mean():.4f}')
    print(f'  Std:  {gnn_outputs["anomaly_score"].std():.4f}')
    print(f'  Min:  {gnn_outputs["anomaly_score"].min():.4f}')
    print(f'  Max:  {gnn_outputs["anomaly_score"].max():.4f}')
    print(f'  >0.7: {(gnn_outputs["anomaly_score"] > 0.7).sum()} ({(gnn_outputs["anomaly_score"] > 0.7).mean()*100:.1f}%)')
    print(f'  >0.5: {(gnn_outputs["anomaly_score"] > 0.5).sum()} ({(gnn_outputs["anomaly_score"] > 0.5).mean()*100:.1f}%)')
    
    # Load embeddings for cluster quality metrics
    embeddings = np.stack(gnn_outputs['embedding'].values)
    cluster_ids = gnn_outputs['cluster_id'].values
    
    from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
    n_clusters = len(np.unique(cluster_ids))
    if n_clusters > 1 and n_clusters < len(embeddings):
        sil = silhouette_score(embeddings, cluster_ids)
        db_idx = davies_bouldin_score(embeddings, cluster_ids)
        ch_idx = calinski_harabasz_score(embeddings, cluster_ids)
        print(f'\nCluster Quality Metrics:')
        print(f'  Silhouette:      {sil:.4f} (higher better, [-1,1])')
        print(f'  Davies-Bouldin:  {db_idx:.4f} (lower better)')
        print(f'  Calinski-Harabasz: {ch_idx:.4f} (higher better)')
else:
    print('No GNN results available')

NameError: name 'gnn_cluster_df' is not defined

## 6. Cross-Model Comparison

In [ ]:
# Cross-Model Comparison (where comparable)
print('='*60)
print('CROSS-MODEL COMPARISON')
print('='*60)

# Note: SARIMA and XGBoost both predict monthly_kg, but on different paradigms
# SARIMA: per-farm time series forecasts
# XGBoost: supervised regression on feature matrix
# GNN: unsupervised clustering/anomaly detection (not directly comparable)

if sarima_df is not None and len(sarima_df) > 0:
    print('\nSARIMA (horizon=1 month, aggregate):')
    h1 = sarima_df[sarima_df['horizon'] == 1]
    print(f'  RMSE: {h1["rmse"].mean():.2f} kg')
    print(f'  R²:   {h1["r2"].mean():.4f}')
    print(f'  MAE:  {h1["mae"].mean():.2f} kg')
    print(f'  MAPE: {h1["mape"].mean():.2f}%')
    print(f'  Farms: {h1["member_no"].nunique()}')

if xgb_pred_df is not None:
    print('\nXGBoost (holdout 2023-2024):')
    rmse = np.sqrt(np.mean((xgb_pred_df['predicted'] - xgb_pred_df['actual'])**2))
    r2 = 1 - np.sum((xgb_pred_df['actual'] - xgb_pred_df['predicted'])**2) / np.sum((xgb_pred_df['actual'] - xgb_pred_df['actual'].mean())**2)
    mae = np.mean(np.abs(xgb_pred_df['predicted'] - xgb_pred_df['actual']))
    mape = np.mean(np.abs((xgb_pred_df['actual'] - xgb_pred_df['predicted']) / xgb_pred_df['actual'])) * 100
    print(f'  RMSE: {rmse:.2f} kg')
    print(f'  R²:   {r2:.4f}')
    print(f'  MAE:  {mae:.2f} kg')
    print(f'  MAPE: {mape:.2f}%')
    print(f'  Samples: {len(xgb_pred_df)}')

# Note about comparability
print('\n⚠️  IMPORTANT: Direct comparison caveats:')
print('  - SARIMA evaluated per-farm on time series (1-month ahead)')
print('  - XGBoost evaluated on row-level holdout (all farms, all months)')
print('  - Different test sets, different paradigms')
print('  - GNN is unsupervised — evaluates cluster quality, not regression')

## 7. Export Results for Reporting

In [ ]:
# Export summary tables for reports
export_dir = BASE_DIR / 'results' / 'benchmark_exports'
export_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

if sarima_df is not None and len(sarima_df) > 0:
    # SARIMA summary by horizon
    sarima_summary = sarima_df.groupby('horizon')[['rmse', 'mse', 'r2', 'mae', 'mape']].agg(['mean', 'std', 'count'])
    sarima_summary.to_csv(export_dir / f'sarima_summary_{timestamp}.csv')
    
    # Per-farm SARIMA
    sarima_farm = sarima_df.groupby('member_no')[['rmse', 'r2', 'mae']].mean().reset_index()
    sarima_farm.to_csv(export_dir / f'sarima_per_farm_{timestamp}.csv', index=False)

if xgb_cv_df is not None:
    # XGBoost CV summary
    xgb_cv_summary = xgb_cv_df[['rmse', 'mse', 'r2', 'mae', 'mape']].agg(['mean', 'std'])
    xgb_cv_summary.to_csv(export_dir / f'xgb_cv_summary_{timestamp}.csv')

if xgb_farm_df is not None:
    xgb_farm_df.to_csv(export_dir / f'xgb_per_farm_{timestamp}.csv', index=False)

if gnn_cluster_df is not None:
    gnn_cluster_df.to_csv(export_dir / f'gnn_clusters_{timestamp}.csv', index=False)

print(f'Exported summary tables to: {export_dir}')

## 8. Advanced: Correlation Between Model Outputs

In [ ]:
# Explore correlations between different model outputs on overlapping farms
if sarima_df is not None and xgb_farm_df is not None:
    # Get per-farm SARIMA metrics (horizon=1)
    sarima_farm = sarima_df[sarima_df['horizon'] == 1].groupby('member_no')[['rmse', 'mae', 'r2']].mean().reset_index()
    sarima_farm.columns = ['ktda_member_no', 'sarima_rmse', 'sarima_mae', 'sarima_r2']
    
    # Merge with XGBoost per-farm metrics
    merged = sarima_farm.merge(xgb_farm_df[['ktda_member_no', 'mae', 'rmse', 'mape']], 
                              on='ktda_member_no', suffixes=('_sarima', '_xgb'))
    
    print(f'Farms with both SARIMA and XGBoost: {len(merged)}')
    
    if len(merged) > 10:
        # Correlation between SARIMA and XGBoost errors
        corr_rmse = merged['sarima_rmse'].corr(merged['rmse'])
        corr_mae = merged['sarima_mae'].corr(merged['mae'])
        print(f'\nCorrelation between SARIMA and XGBoost errors:')
        print(f'  RMSE correlation: {corr_rmse:.4f}')
        print(f'  MAE correlation:  {corr_mae:.4f}')
        
        # Scatter plot
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].scatter(merged['sarima_rmse'], merged['rmse'], alpha=0.6)
        axes[0].plot([0, max(merged['sarima_rmse'].max(), merged['rmse'].max())], 
                    [0, max(merged['sarima_rmse'].max(), merged['rmse'].max())], 'r--')
        axes[0].set_xlabel('SARIMA RMSE (kg)')
        axes[0].set_ylabel('XGBoost RMSE (kg)')
        axes[0].set_title(f'Per-Farm RMSE Correlation (r={corr_rmse:.3f})')
        
        axes[1].scatter(merged['sarima_mae'], merged['mae'], alpha=0.6)
        axes[1].plot([0, max(merged['sarima_mae'].max(), merged['mae'].max())], 
                    [0, max(merged['sarima_mae'].max(), merged['mae'].max())], 'r--')
        axes[1].set_xlabel('SARIMA MAE (kg)')
        axes[1].set_ylabel('XGBoost MAE (kg')
        axes[1].set_title(f'Per-Farm MAE Correlation (r={corr_mae:.3f})')
        
        plt.tight_layout()
        plt.show()
        
        # Farms where models disagree
        merged['rmse_diff'] = merged['sarima_rmse'] - merged['rmse']
        merged['mae_diff'] = merged['sarima_mae'] - merged['mae']
        
        print('\nFarms where SARIMA outperforms XGBoost (lower RMSE):')
        better_sarima = merged.nsmallest(10, 'rmse_diff')
        display(better_sarima[['ktda_member_no', 'sarima_rmse', 'rmse', 'rmse_diff']])
        
        print('\nFarms where XGBoost outperforms SARIMA (lower RMSE):')
        better_xgb = merged.nlargest(10, 'rmse_diff')
        display(better_xgb[['ktda_member_no', 'sarima_rmse', 'rmse', 'rmse_diff']])
else:
    print('Need both SARIMA and XGBoost results for correlation analysis')